# 09 - Spark Connect

Run Spark from this notebook against the in-stack **Spark Connect** sidecar.
There is no Spark distribution in this image — `pyspark-client` is a thin gRPC
client (~1.5 MB, no JVM). The driver runs on the `spark-connect` container,
executors on `spark-worker`, and results stream back here.

`SPARK_REMOTE` is injected by compose (default `sc://spark-connect:15002`).
Requires `SPARK_SOURCE != disabled` — enable it with
`./start.sh --spark-source container`. Override `SPARK_REMOTE` to point at a
remote/managed Spark Connect endpoint (e.g. EMR Serverless) instead.

In [ ]:
import os
from pyspark.sql import SparkSession

spark_remote = os.environ.get("SPARK_REMOTE", "sc://spark-connect:15002").strip()

try:
    spark = SparkSession.builder.remote(spark_remote).getOrCreate()
    print("connected:", spark_remote, "| Spark", spark.version)
except Exception as exc:  # noqa: BLE001
    raise RuntimeError(
        f"Could not reach Spark Connect at {spark_remote!r}: {exc}\n"
        "Is Spark enabled? Re-run ./start.sh --spark-source container (the "
        "spark-connect sidecar's JVM also lags spark-master by 20-60s on a "
        "cold start). The client's pyspark-client version must match the "
        "server's Spark version (4.1.2)."
    ) from exc

## DataFrame + SQL
These run on the remote driver, not in this kernel.

In [ ]:
from pyspark.sql.functions import col

df = spark.range(10).withColumn("squared", col("id") * col("id"))
df.show()
print("count:", df.count())
spark.sql("SELECT 1 + 1 AS result").show()

## MinIO (s3a) round-trip
Storage credentials live on the **server** (the sidecar's conf), so `s3a://`
just works from here — no keys in the notebook. Writes also land in the Spark
History Server automatically (`spark.eventLog` is set server-side).

In [ ]:
path = "s3a://spark-history/_notebook09_probe"
spark.range(5).write.mode("overwrite").parquet(path)
print("rows read back:", spark.read.parquet(path).count())

In [ ]:
spark.stop()